# Config

In [1]:
!git config --global --add safe.directory /tmp/Repository/VRID_language_proyect

In [2]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


# 1) Split dataset

In [ ]:
from utils.dataset import get_dataset_to_split, split_dataset
import pandas as pd
import numpy as np
import os

#Save data
path = "/tmp/final_project"
filepath = os.path.join(path, "datasets/features.csv")
df=pd.read_csv(filepath)

feat_col = "Desafío País"
df = get_dataset_to_split(df, feat_col)

#Eliminar elementos indefinidos 
df = df[df["Desafío País"].notna()]

#Eliminar duplicados
df = df.drop_duplicates("Código VRID")

#Split data and save idx
ids = np.array(df["Código VRID"])
labels = np.array(df["Desafío País"])
savepath = os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
split_dataset(savepath, ids, labels)


Test size: 247
Fold 0 - Val size: 330
Archivo guardado exitosamente en /tmp/final_project/dataSplits/desafios/train_test_ids_3folds.json


# 1) TF-IDF

### Carga y entrenamiento

In [36]:
import json
import pandas as pd

#Ruta de lectura
path = "/tmp/final_project"

#Lectura de index de separacion de conjuntos train/test
filepath=os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
filepath=os.path.join(path, "datasets/data_translated.csv")
df = pd.read_csv(filepath)


In [42]:
from utils.dataset import binarize_labels
import numpy as np

comb_desafios = [["1", "1,2,3"],
                ["2", "2.4", "1,2,3"],
                ["3", "3.4", "1,2,3"],
                ["4", "2.4", "3.4"]]

#Crear one-ot
for des in comb_desafios: 
    labels = np.isin(df["Desafío País"], des)
    labels = np.where(labels, 1, 0).astype(int)
    df[str(des)] = labels

In [48]:
from sklearn.preprocessing import LabelEncoder
from utils.dataset import gen_dataset_select_cols
from models.TIFD import gen_TFID_vectors
import numpy as np
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model
from utils.dataset import CvCustom
from collections import Counter
from utils.mlflow import eval_model
import warnings

warnings.filterwarnings(
    "ignore",
    message="The objective has been evaluated at point",
    category=UserWarning,
    module="skopt.optimizer.optimizer"
)

for des in comb_desafios:
    #Columnas a seleccionar para clasificación
    cols = ["Titulo_trad", "keywords_trad", "Resumen_trad"]

    #Lectura de codigos VRID Test
    codes_test = dataset_index["Test"]
    X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, 
                                                    test_col=str(des))

    #Lectura de codigos VRID Train
    codes_train = dataset_index["kfolds"]
    codes_train = np.array([i for fold in codes_train for i in fold])
    X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols,
                                                        test_col=str(des))
    df_decode = df_train[["idx", "Código VRID"]]

    # Codificación de labels
    le = LabelEncoder()
    y_train = le.fit_transform(y_train)
    y_test = le.transform(y_test)

    #Creacion de vectores TFID
    X_train, X_test = gen_TFID_vectors(X_train, X_test)
    print(X_train.shape, X_test.shape)

    # 1. Elegir modelos a probar
    model_keys = [
        'LogisticRegression',
        'RandomForestClassifier',
        'XGBClassifier',
        'SVC',
    ]

    # 2. Obtener el diccionario de modelos y parámetros
    est_params_dict = get_est_params_dict(model_keys)
    print("📊 train:", Counter(y_train))
    print("📊 test:", Counter(y_test))

    # 3. Ejecutar entrenamiento, validación y test con tus funciones
    n_iter=20
    sample_weight_On=True
    scoring='f1_macro'
    split_idx_path = os.path.join(path, "dataSplits/desafios/train_test_ids_3folds.json")
    results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode, split_idx_path), 
                                                    n_iter=n_iter, sample_weight_On = sample_weight_On)

    best_model = select_best_model(results_val, models_dicc)

    # 4. Mostrar resultados
    print("\n🔍 Validación:")
    for model, metrics in results_val.items():
        print(f"{model}: {metrics}")

    # Métricas por idioma
    lang_es = df_test["Español"]
    for name, model in models_dicc.items():
        print(name)
        results, preds=eval_model(model, X_test, y_test, lang_es)
        print(results)

(987, 17354) (247, 17354)
📊 train: Counter({0: 887, 1: 100})
📊 test: Counter({0: 222, 1: 25})
(987,)
LogisticRegression
Compute sw
RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.68, 'std_test_score': 0.02}
RandomForestClassifier: {'mean_test_score': 0.62, 'std_test_score': 0.03}
XGBClassifier: {'mean_test_score': 0.69, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.67, 'std_test_score': 0.05}
LogisticRegression
{'accuracy': 0.8744939271255061, 'f1_macro': 0.6828743010975358, 'cm': array([[204,  18],
       [ 13,  12]]), 'precision': 0.4, 'recall': 0.48, 'f1_es': 0.8765227021040976, 'f1_en': 0.8618986091054192, 'cm_es': array([[99, 15],
       [ 3, 12]]), 'cm_en': array([[105,   3],
       [ 10,   0]])}
RandomForestClassifier
{'accuracy': 0.8987854251012146, 'f1_macro': 0.694382578569661, 'cm': array([[212,  10],
       [ 15,  10]]), 'precision': 0.5, 'recall': 0.4, 'f1_es': 0.8968878248974008, 'f1_e